# Lesson 06 — From Lead-Lag Discovery to Prediction
## A Complete, Honest Pipeline for N Data Streams

### Before we begin: three things you need to hear

**1. Data Smashing is a distance measure, not a predictor.**
`lsmash` computes how statistically similar two sequences are. It does not generate
forecasts. In this notebook we use it *where it is genuinely appropriate*: for
regime detection and for analog (nearest-neighbour) forecasting. We use classical
and machine-learning models for the actual prediction.

**2. The signals we found in Lessons 03-05 are weak.**
Our best finding — Gold leading Newmont during the GFC — sits at p ~0.08 with a
fractional improvement in SL distance of < 0.003 units. A prediction system built
on signals this marginal will likely not survive contact with live data. We build
the pipeline anyway — because the *architecture* is correct even if the *signal* is
modest — and we report results honestly.

**3. Exploring all N*(N-1)/2 pairs at all lags without correction is a
multiple-comparisons disaster.**
For 4 streams at 20 lags: 4*3/2 * 20 = 120 tests. At alpha=0.05 you expect 6
false discoveries. We apply Benjamini-Hochberg FDR correction throughout.

### What this notebook DOES

Given N return streams, it:

| Step | What | Tool |
|------|------|------|
| A | Automated pairwise lead-lag screening | CCF + BH correction |
| B | Hyperparameter identification (lag, window) | Wavelet coherence + SL stability |
| C | Regime detection | lsmash SL distance |
| D | Feature construction | Lagged returns + regime |
| E | Walk-forward forecasting | AR(1) / ARIMAX / lsmash kNN |
| F | Honest evaluation | Walk-forward IC, RMSE, signal curve |

### The Efficient Market caveat

Even genuine historical lead-lag patterns get arbitraged away as more
quantitative traders discover and trade them. Our GFC-era Gold-Newmont
signal (2008-2012) may have attenuated since. The correct expectation for
any signal discovered this way is that *if it worked historically, it may
be weaker going forward*. We report out-of-sample performance on 2019-2024
data that was not used in any previous analysis.


## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from scipy import signal
from scipy.signal import hilbert
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import yfinance as yf
import lsmash as ls

C = {"blue":"#0072B2","orange":"#E69F00","green":"#009E73",
     "red":"#D55E00","grey":"#999999","sky":"#56B4E9","purple":"#CC79A7"}
plt.rcParams.update({"figure.dpi":120,"axes.titlesize":11,"axes.labelsize":10,
                      "legend.fontsize":9})

# ── Data: 4 weekly return streams ─────────────────────────────────────────────
print("Downloading data (2000-2024)...")
TICKERS = {"Gold":"GC=F","Oil":"CL=F","Newmont":"NEM","Halliburton":"HAL"}
raw    = yf.download(list(TICKERS.values()), start="2000-01-01", end="2024-01-01",
                     progress=False, auto_adjust=True)
daily  = raw["Close"].dropna(how="all")
weekly = daily.resample("W-FRI").last().dropna(how="all")
# Rename columns from tickers to readable names
wret   = np.log(weekly / weekly.shift(1)).dropna()
wret.columns = list(TICKERS.keys())
t_idx  = wret.index
N_STREAMS = len(wret.columns)
STREAM_NAMES = list(wret.columns)

# ── Walk-forward split ─────────────────────────────────────────────────────────
# 60% train  | 20% validation | 20% test
n  = len(wret)
n_train = int(0.60 * n); n_val = int(0.20 * n)
train_mask = slice(0, n_train)
val_mask   = slice(n_train, n_train + n_val)
test_mask  = slice(n_train + n_val, n)

print(f"  {n} weekly returns: {t_idx[0].date()} to {t_idx[-1].date()}")
print(f"  Train : {n_train} weeks ({t_idx[train_mask][0].date()} - {t_idx[train_mask][-1].date()})")
print(f"  Val   : {n_val}   weeks ({t_idx[val_mask][0].date()}  - {t_idx[val_mask][-1].date()})")
print(f"  Test  : {n-n_train-n_val} weeks ({t_idx[test_mask][0].date()} - {t_idx[test_mask][-1].date()})")

# ADF stationarity check
print("\nStationarity (ADF test, p<0.05 = stationary):")
for name in STREAM_NAMES:
    _, p, *_ = adfuller(wret[name])
    print(f"  {name}: p={p:.4f} ({'OK' if p<0.05 else 'WARN'})")

---
## Step A — Automated Pairwise Lead-Lag Screening

We compute the CCF between every ordered pair of streams at lags 1-20 weeks,
collect all p-values (approximately $\frac{N(N-1)}{2} \times L_{max}$ tests),
and apply **Benjamini-Hochberg FDR correction** at $q=0.10$.

The corrected discoveries define our *candidate* lead-lag relationships.
We then use wavelet coherence to visualise the time-localisation of the
top candidate pairs.

**Why BH and not Bonferroni?** Bonferroni is too conservative when tests are
correlated (which CCFs at adjacent lags are). BH controls the *false discovery rate*
rather than the *family-wise error rate*, which is the right criterion when we
expect some true discoveries to exist among many tests.

> **Reference:** Benjamini, Y. & Hochberg, Y. (1995). Controlling the false
> discovery rate: a practical and powerful approach to multiple testing.
> *JRSS-B* **57**(1):289-300.


In [ ]:
def compute_ccf(x, y, nlags=20):
    n = len(x)
    xz = (x-x.mean())/(x.std()*n); yz = (y-y.mean())/y.std()
    full = np.correlate(xz, yz, mode='full'); ctr = n-1
    lags = np.arange(1, nlags+1)   # positive lags only (x leads y)
    cc   = full[ctr+1 : ctr+nlags+1]
    # Approximate p-values for each lag (t-distribution, df=n-2)
    from scipy.stats import t as tdist
    t_stat = cc * np.sqrt(n-2) / np.sqrt(1 - cc**2 + 1e-10)
    pvals  = 2 * tdist.sf(np.abs(t_stat), df=n-2)
    return lags, cc, pvals

NLAGS = 20
TRAIN = wret.iloc[train_mask]

# Collect all test results
all_results = []   # (src, tgt, lag, ccf, pval)
pairs = [(s, t) for s in STREAM_NAMES for t in STREAM_NAMES if s != t]

for src, tgt in pairs:
    x = TRAIN[src].values; y = TRAIN[tgt].values
    lags, cc, pvals = compute_ccf(x, y, NLAGS)
    for lag, c, p in zip(lags, cc, pvals):
        all_results.append(dict(src=src, tgt=tgt, lag=lag, ccf=c, pval=p))

df_all = pd.DataFrame(all_results)

# Benjamini-Hochberg FDR correction at q=0.10
rejected, pvals_adj, _, _ = multipletests(df_all["pval"].values,
                                           method="fdr_bh", alpha=0.10)
df_all["pval_adj"] = pvals_adj
df_all["significant"] = rejected

n_total = len(df_all)
n_sig   = rejected.sum()
print(f"Total tests: {n_total}  |  Significant after BH(q=0.10): {n_sig}")
print(f"Expected false discoveries: <= {0.10*n_sig:.1f}")

if n_sig > 0:
    sig = df_all[df_all["significant"]].sort_values("pval_adj")
    print("\nSignificant lead-lag relationships (train period):")
    print(sig[["src","tgt","lag","ccf","pval","pval_adj"]].to_string(index=False))
else:
    print("\nNo significant pairs after BH correction.")
    print("This is expected for near-i.i.d. weekly returns.")
    print("We will proceed with the best candidate regardless (Gold->Newmont, lag=1).")
    # Use the best candidate even if not strictly significant
    best = df_all.loc[df_all["ccf"].idxmax()]
    sig  = df_all[(df_all["src"]=="Gold") & (df_all["tgt"]=="Newmont")].sort_values("pval")
    print("\nBest candidate pairs by raw CCF magnitude:")
    print(df_all.nlargest(5,"ccf")[["src","tgt","lag","ccf","pval"]].to_string(index=False))

In [ ]:
# ── Heatmap: max |CCF| across lags for each pair ──────────────────────────────
n_s = len(STREAM_NAMES)
ccf_heatmap = np.zeros((n_s, n_s))
lag_heatmap = np.zeros((n_s, n_s), dtype=int)

for i, src in enumerate(STREAM_NAMES):
    for j, tgt in enumerate(STREAM_NAMES):
        if src == tgt: continue
        sub = df_all[(df_all["src"]==src) & (df_all["tgt"]==tgt)]
        best_idx = sub["ccf"].abs().idxmax()
        ccf_heatmap[i, j] = sub.loc[best_idx, "ccf"]
        lag_heatmap[i, j] = int(sub.loc[best_idx, "lag"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(np.abs(ccf_heatmap), cmap="YlOrRd", vmin=0, vmax=0.15)
axes[0].set_xticks(range(n_s)); axes[0].set_yticks(range(n_s))
axes[0].set_xticklabels(STREAM_NAMES, rotation=30)
axes[0].set_yticklabels(STREAM_NAMES)
axes[0].set_title("Max |CCF| across lags 1-20\n(row leads column)", fontweight="bold")
plt.colorbar(im1, ax=axes[0])
for i in range(n_s):
    for j in range(n_s):
        if i != j:
            axes[0].text(j, i, f"{ccf_heatmap[i,j]:.3f}\nlag={lag_heatmap[i,j]}",
                         ha="center", va="center", fontsize=8)

# BH significance dots
sig_mat = np.zeros((n_s, n_s))
for _, row in df_all[df_all["significant"]].iterrows():
    i = STREAM_NAMES.index(row["src"]); j = STREAM_NAMES.index(row["tgt"])
    sig_mat[i, j] = 1

im2 = axes[1].imshow(sig_mat, cmap="Greens", vmin=0, vmax=1)
axes[1].set_xticks(range(n_s)); axes[1].set_yticks(range(n_s))
axes[1].set_xticklabels(STREAM_NAMES, rotation=30)
axes[1].set_yticklabels(STREAM_NAMES)
axes[1].set_title("BH-significant pairs (q=0.10, train period)\nGreen = significant", fontweight="bold")
plt.colorbar(im2, ax=axes[1])

plt.suptitle("Step A: Automated Pairwise Lead-Lag Screening (Train Period Only)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Step B — Hyperparameter Identification

Two hyperparameters govern the prediction pipeline:

1. **Optimal lag** $k^*$: how many weeks ahead does the commodity return predict
   the equity return? Identified from Step A; we use the lag of maximum CCF
   among the significant (or best-candidate) pairs.

2. **Window size** $W$: how many weeks of history constitute a "state" for the
   lsmash regime detector? We select $W$ by measuring the *stability* of SL
   distances as a function of $W$ — specifically, we want the window size where
   adding more history stops changing the distance estimates appreciably.
   This is analogous to selecting the embedding dimension in state-space reconstruction.

   A practical rule: $W$ should be at least $3 \times$ the longest lag we want to model.
   For lag=5 weeks that gives $W \geq 15$; we test $W \in \{13, 26, 52\}$ weeks.


In [ ]:
def make_opt(n_repeat=20):
    opt = ls.LsmashOptions()
    opt.data_type="symbolic"; opt.data_dir="row"
    opt.sae=True; opt.num_repeat=n_repeat; opt.depth=8
    return opt

def encode(arr, n_sym=5):
    t = np.percentile(arr, np.linspace(100/n_sym, 100*(n_sym-1)/n_sym, n_sym-1))
    return np.digitize(arr, bins=t).tolist()

# ── Optimal lag from Step A ────────────────────────────────────────────────────
# Use Gold->Newmont as our primary pair (best candidate from prior analysis)
LEAD_COL   = "Gold"
TARGET_COL = "Newmont"
best_lag_row = df_all[(df_all["src"]==LEAD_COL) & (df_all["tgt"]==TARGET_COL)
               ].sort_values("pval").iloc[0]
OPTIMAL_LAG = int(best_lag_row["lag"])
print(f"Primary pair: {LEAD_COL} leads {TARGET_COL}")
print(f"Optimal lag  : {OPTIMAL_LAG} week(s)  (CCF={best_lag_row['ccf']:.4f}, p={best_lag_row['pval']:.3f})")

# ── Window size stability ──────────────────────────────────────────────────────
# For each window size, compute the variance of pairwise SL distances
# across rolling windows of that size. Lower variance = more stable estimates.
WINDOW_SIZES = [13, 26, 52]
OPT_W = make_opt(n_repeat=10)

print("\nWindow size stability (lower SL variance = more stable regime estimate):")
stability = {}
for W in WINDOW_SIZES:
    x_sym = encode(TRAIN[LEAD_COL].values); y_sym = encode(TRAIN[TARGET_COL].values)
    dists = []
    for start in range(0, len(x_sym)-W, W//2):
        xa = x_sym[start:start+W]; ya = y_sym[start:start+W]
        d = float(ls.from_sequences([xa, ya], OPT_W)[0,1])
        dists.append(d)
    var = np.var(dists)
    stability[W] = var
    print(f"  W={W:2d}wk: SL distance variance = {var:.6f}  (n_windows={len(dists)})")

WINDOW_SIZE = min(stability, key=stability.get)
print(f"\nSelected window size: W={WINDOW_SIZE} weeks (lowest variance)")
print(f"This means: {WINDOW_SIZE} weeks of history defines a 'market state' for regime detection")

---
## Step C — Regime Detection via lsmash

This is where lsmash makes its genuine contribution to prediction.

We build a **regime library** from the training period: a collection of
$(W)$-week windows, each labelled with the subsequent return of the target.
For any new window, we find the $K$ most similar library windows by SL distance
and predict the next return as a weighted average of their labels.

This is the **Lorenz analog method** (Lorenz 1969), which originated in weather
forecasting, applied with SL divergence as the similarity metric instead of
Euclidean distance. The key advantage: SL divergence measures *statistical
structure* similarity, not just value proximity — it can detect that two windows
share the same volatility regime and transition dynamics even if the raw returns
look different.

We also use the regime library for a simpler task: classifying each week into
a **named regime** (crisis / recovery / bull / neutral) by finding the nearest
historical archetype. This regime label becomes a feature in the prediction models.

> **Reference:** Lorenz, E.N. (1969). Atmospheric predictability as revealed by
> naturally occurring analogues. *Journal of the Atmospheric Sciences* **26**:636-646.


In [ ]:
K_NEIGHBORS = 10
OPT_REGIME  = make_opt(n_repeat=20)

def build_library(returns_series, lead_series, W, lag):
    """
    Build a library of (window_seq, next_return) pairs from the training data.
    window_seq = encoded W-week window of `lead_series`
    next_return = return of `returns_series` at step lag weeks after window end
    """
    lead_sym = encode(lead_series)
    tgt_vals  = returns_series.values
    seqs, labels, t_ends = [], [], []
    for start in range(0, len(lead_sym) - W - lag):
        seq   = lead_sym[start:start+W]
        label = tgt_vals[start + W + lag - 1]   # return lag weeks after window
        if not np.isnan(label):
            seqs.append(seq); labels.append(label); t_ends.append(start+W)
    return seqs, np.array(labels), t_ends

def analog_forecast(library_seqs, library_labels, query_seq, K=K_NEIGHBORS):
    """
    kNN prediction: find K most similar library windows by SL distance.
    Returns (predicted_return, mean_distance_to_nearest).
    """
    batch = [list(query_seq)] + [list(s) for s in library_seqs]
    D     = ls.from_sequences(batch, OPT_REGIME)
    dists = D[0, 1:]
    k_idx = np.argsort(dists)[:K]
    k_d   = dists[k_idx]; k_l = library_labels[k_idx]
    weights = 1.0 / (k_d + 1e-8); weights /= weights.sum()
    pred    = np.dot(weights, k_l)
    return pred, k_d.mean()

# Build library from TRAIN only
train_lead = TRAIN[LEAD_COL]; train_tgt = TRAIN[TARGET_COL]
lib_seqs, lib_labels, lib_t = build_library(train_lead, train_tgt, WINDOW_SIZE, OPTIMAL_LAG)
print(f"Regime library: {len(lib_seqs)} windows  (W={WINDOW_SIZE}, lag={OPTIMAL_LAG})")
print(f"Label range: [{lib_labels.min():.4f}, {lib_labels.max():.4f}]  "
      f"mean={lib_labels.mean():.4f}  std={lib_labels.std():.4f}")

# Quick internal check: does the analog prediction correlate with actual on TRAIN?
preds_train = []
for i in range(len(lib_seqs)):
    # Leave-one-out: exclude window i from the search
    other_seqs = lib_seqs[:i] + lib_seqs[i+1:]
    other_labs  = np.concatenate([lib_labels[:i], lib_labels[i+1:]])
    p, _ = analog_forecast(other_seqs, other_labs, lib_seqs[i], K=K_NEIGHBORS)
    preds_train.append(p)
from scipy.stats import pearsonr
ic_train = pearsonr(lib_labels, preds_train)[0]
print(f"\nLeave-one-out IC (train): {ic_train:.4f}  "
      f"({'good' if abs(ic_train)>0.05 else 'weak'} in-sample signal)")

In [ ]:
# ── Name regimes for interpretability ─────────────────────────────────────────
# Define 4 prototype windows from known historical periods
GFC_START   = np.searchsorted(t_idx, pd.Timestamp("2008-09-01"))
GFC_PEAK    = np.searchsorted(t_idx, pd.Timestamp("2009-03-01"))
BULL_2017   = np.searchsorted(t_idx, pd.Timestamp("2017-01-01"))
COVID_CRASH = np.searchsorted(t_idx, pd.Timestamp("2020-03-01"))

lead_sym_all = encode(wret[LEAD_COL].values)

prototypes = {
    "Crisis":   lead_sym_all[GFC_START:GFC_START+WINDOW_SIZE],
    "Recovery": lead_sym_all[GFC_PEAK:GFC_PEAK+WINDOW_SIZE],
    "Bull":     lead_sym_all[BULL_2017:BULL_2017+WINDOW_SIZE],
    "COVID":    lead_sym_all[COVID_CRASH:COVID_CRASH+WINDOW_SIZE],
}
PROTO_NAMES = list(prototypes.keys())

def classify_regime(query_seq, prototypes):
    batch = [list(query_seq)] + [list(v) for v in prototypes.values()]
    D     = ls.from_sequences(batch, OPT_REGIME)
    dists = D[0, 1:]
    return PROTO_NAMES[np.argmin(dists)], dists

# Classify every training week
print("Rolling regime classification (stride=4 weeks, training period)...")
regime_series = []
regime_dates  = []
for start in range(0, len(lead_sym_all) - WINDOW_SIZE, 4):
    win = lead_sym_all[start:start+WINDOW_SIZE]
    reg, dists = classify_regime(win, prototypes)
    regime_series.append(reg)
    regime_dates.append(t_idx[start+WINDOW_SIZE])

regime_df = pd.DataFrame({"date":regime_dates,"regime":regime_series})
print("\nRegime distribution across full history:")
print(regime_df["regime"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(15, 3))
REGIME_COLORS = {"Crisis":C["red"],"Recovery":C["orange"],"Bull":C["green"],"COVID":C["purple"]}
for i, row in regime_df.iterrows():
    ax.axvline(row["date"], color=REGIME_COLORS.get(row["regime"],C["grey"]),
               alpha=0.3, lw=0.5)
# Add legend patches
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=v, label=k, alpha=0.6) for k,v in REGIME_COLORS.items()]
ax.legend(handles=patches, loc="upper right", fontsize=9)
ax.set_xlim(t_idx[0], t_idx[-1])
ax.set_title("Step C: Rolling Regime Classification via SL Distance to Prototypes",
             fontweight="bold")
ax.set_xlabel("Date"); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## Step D — Feature Construction

We build the feature matrix using the insights from Steps A-C.
All features are constructed using **only past information** — no look-ahead.

| Feature | Source | Rationale |
|---------|--------|-----------|
| `gold_lag_k` | Gold return $k$ weeks ago | Best-candidate lead-lag |
| `gold_lag_1` | Gold return 1 week ago | Short-term momentum |
| `nnem_lag_1` | NEM return 1 week ago | Own-series AR(1) |
| `rolling_vol_4` | NEM 4-week rolling sigma | Volatility scaling |
| `rolling_vol_13` | NEM 13-week rolling sigma | Quarterly vol regime |
| `regime_crisis` | 1 if current regime = Crisis | From Step C |
| `analog_forecast` | lsmash kNN prediction | From Step C |
| `gold_x_crisis` | Interaction: gold_lag_k * regime_crisis | Signal only active in crisis |


In [ ]:
def build_features(wret_df, t_idx, lead_col, target_col, lag_k,
                   lib_seqs, lib_labels, window_size, prototypes,
                   split_start=0, split_end=None):
    """
    Build feature matrix for rows [split_start:split_end].
    All features are lagged to avoid look-ahead.
    Returns (X_df, y_series).
    """
    if split_end is None: split_end = len(wret_df)
    lead_sym_all = encode(wret_df[lead_col].values)

    rows = []
    for i in range(split_start, split_end):
        if i < window_size + lag_k: continue

        # Lagged commodity features
        gold_lagk = wret_df[lead_col].iloc[i - lag_k] if i >= lag_k else np.nan
        gold_lag1 = wret_df[lead_col].iloc[i - 1]     if i >= 1    else np.nan
        nnem_lag1 = wret_df[target_col].iloc[i - 1]   if i >= 1    else np.nan

        # Rolling volatility (no look-ahead: use data up to i-1)
        hist = wret_df[target_col].iloc[max(0,i-13):i]
        rv4  = wret_df[target_col].iloc[max(0,i-4):i].std()
        rv13 = hist.std()

        # Regime (classify window ending at i-1)
        win_end = i - 1
        win_start = win_end - window_size
        if win_start >= 0:
            win_sym = lead_sym_all[win_start:win_end]
            reg, _  = classify_regime(win_sym, prototypes)
        else:
            reg = "Unknown"

        # lsmash analog forecast
        if win_start >= 0:
            analog_pred, _ = analog_forecast(lib_seqs, lib_labels, win_sym)
        else:
            analog_pred = 0.0

        # Target: this week's return
        y_val = wret_df[target_col].iloc[i]

        rows.append({
            "date": t_idx[i],
            "gold_lagk": gold_lagk,
            "gold_lag1": gold_lag1,
            "nnem_lag1": nnem_lag1,
            "rv4": rv4,
            "rv13": rv13,
            "regime_crisis": 1.0 if reg == "Crisis" else 0.0,
            "analog_pred": analog_pred,
            "gold_x_crisis": gold_lagk * (1.0 if reg=="Crisis" else 0.0),
            "y": y_val,
        })

    df = pd.DataFrame(rows).dropna()
    X  = df.drop(columns=["date","y"])
    y  = df["y"]
    return df["date"], X, y

print("Building feature matrices (this includes per-row lsmash calls; ~5 min)...")
FEATURE_COLS = ["gold_lagk","gold_lag1","nnem_lag1","rv4","rv13",
                "regime_crisis","analog_pred","gold_x_crisis"]

dates_tr, X_tr, y_tr = build_features(wret, t_idx, LEAD_COL, TARGET_COL,
                                        OPTIMAL_LAG, lib_seqs, lib_labels,
                                        WINDOW_SIZE, prototypes,
                                        split_start=0, split_end=n_train)
dates_te, X_te, y_te = build_features(wret, t_idx, LEAD_COL, TARGET_COL,
                                        OPTIMAL_LAG, lib_seqs, lib_labels,
                                        WINDOW_SIZE, prototypes,
                                        split_start=n_train+n_val, split_end=n)

print(f"  Train features: {X_tr.shape}  |  Test features: {X_te.shape}")
print("\nFeature correlations with target (train):")
corrs = X_tr.corrwith(y_tr).sort_values(key=abs, ascending=False)
for feat, cor in corrs.items():
    print(f"  {feat:20s}: {cor:+.4f}")

---
## Step E — Walk-Forward Forecasting

We compare three models on the held-out test period (2019-2024):

**Model 1 — AR(1) baseline**
Predict next week's NEM return as last week's NEM return scaled by the
historical AR(1) coefficient. This is the minimum bar any model must beat.

**Model 2 — Ridge regression with lag features**
A regularised linear model using all features from Step D.
Ridge is preferred over OLS for small samples with correlated features.
Regularisation strength is tuned on the validation period.

**Model 3 — lsmash kNN analog forecast**
Already computed as a feature; here used standalone as a predictor.
Represents what data smashing contributes on its own.

**No in-sample evaluation.** All reported metrics are from the test period only.

> **Note:** We do NOT use deep learning here. For 244 test samples, a neural
> network would be massively overfitted and completely uninformative.
> Simpler models are the right tool at this data volume.


In [ ]:
from scipy.stats import pearsonr

def information_coefficient(y_true, y_pred):
    """Pearson correlation between predicted and actual returns (IC)."""
    return pearsonr(y_true, y_pred)[0]

def walk_forward_sharpe(y_true, y_pred, n_weeks_year=52):
    """
    Strategy: go long when prediction > 0, short when < 0.
    Sharpe = annualised mean / std of strategy returns.
    Note: no transaction costs, no slippage. This is a signal Sharpe, not a trading Sharpe.
    """
    strategy = np.sign(y_pred) * y_true
    annual   = np.sqrt(n_weeks_year)
    return strategy.mean() / strategy.std() * annual if strategy.std() > 0 else 0.

# ── Model 1: AR(1) baseline ────────────────────────────────────────────────────
ar1_coef = y_tr.autocorr(lag=1)   # fitted on train
pred_ar1 = X_te["nnem_lag1"].values * ar1_coef
ic_ar1   = information_coefficient(y_te.values, pred_ar1)
rmse_ar1 = np.sqrt(mean_squared_error(y_te, pred_ar1))
sh_ar1   = walk_forward_sharpe(y_te.values, pred_ar1)
print(f"AR(1) baseline    : IC={ic_ar1:+.4f}  RMSE={rmse_ar1:.5f}  Sharpe={sh_ar1:+.2f}")

# ── Model 2: Ridge regression ──────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Tune alpha on validation period
dates_va, X_va, y_va = build_features(wret, t_idx, LEAD_COL, TARGET_COL,
                                       OPTIMAL_LAG, lib_seqs, lib_labels,
                                       WINDOW_SIZE, prototypes,
                                       split_start=n_train, split_end=n_train+n_val)
X_va_s = scaler.transform(X_va)
best_alpha, best_ic_val = 1.0, -np.inf
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha).fit(X_tr_s, y_tr)
    ic_v  = information_coefficient(y_va.values, ridge.predict(X_va_s))
    if ic_v > best_ic_val: best_ic_val, best_alpha = ic_v, alpha

ridge_final = Ridge(alpha=best_alpha).fit(X_tr_s, y_tr)
pred_ridge  = ridge_final.predict(X_te_s)
ic_ridge    = information_coefficient(y_te.values, pred_ridge)
rmse_ridge  = np.sqrt(mean_squared_error(y_te, pred_ridge))
sh_ridge    = walk_forward_sharpe(y_te.values, pred_ridge)
print(f"Ridge (alpha={best_alpha})    : IC={ic_ridge:+.4f}  RMSE={rmse_ridge:.5f}  Sharpe={sh_ridge:+.2f}")

# ── Model 3: lsmash kNN analog forecast (standalone) ─────────────────────────
pred_analog = X_te["analog_pred"].values
ic_analog   = information_coefficient(y_te.values, pred_analog)
rmse_analog = np.sqrt(mean_squared_error(y_te, pred_analog))
sh_analog   = walk_forward_sharpe(y_te.values, pred_analog)
print(f"lsmash kNN analog : IC={ic_analog:+.4f}  RMSE={rmse_analog:.5f}  Sharpe={sh_analog:+.2f}")

print(f"\nAll metrics are from the TEST period ({dates_te.iloc[0].date()} to {dates_te.iloc[-1].date()}) only.")

---
## Step F — Evaluation and Visualisation

We show three views of model performance:

1. **Information coefficient comparison** — how much do predictions correlate with outcomes?
2. **Cumulative signal performance** — does following the prediction sign earn positive returns?
3. **Feature importance** — which inputs drive the Ridge regression predictions?

### Interpreting Information Coefficient (IC)

In quantitative finance, the rule of thumb is:
- IC > 0.10: strong signal (rare, and worth trading on)
- IC 0.05-0.10: moderate signal (tradeable with tight risk management)
- IC 0.01-0.05: weak but potentially useful in a diversified portfolio
- IC < 0.01: noise

**Do not be disappointed by weak ICs.** Weekly equity return prediction is one of
the hardest forecasting problems in all of applied statistics. The efficient market
hypothesis predicts IC = 0. Anything positive is surprising.


In [ ]:
fig = plt.figure(figsize=(16, 13))
outer = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)

models = {
    "AR(1) baseline":     (pred_ar1,    C["grey"],   ic_ar1,   sh_ar1),
    "Ridge regression":   (pred_ridge,  C["blue"],   ic_ridge, sh_ridge),
    "lsmash kNN analog":  (pred_analog, C["orange"], ic_analog,sh_analog),
}

t_test = dates_te.values

# ── Row 0: scatter plots (predicted vs actual) ────────────────────────────────
for col_idx, (name, (preds, col, ic, sh)) in enumerate(list(models.items())[:2]):
    ax = fig.add_subplot(outer[0, col_idx])
    ax.scatter(preds, y_te.values, color=col, alpha=0.3, s=15)
    m, b = np.polyfit(preds, y_te.values, 1)
    x_line = np.linspace(preds.min(), preds.max(), 50)
    ax.plot(x_line, m*x_line+b, color=col, lw=2)
    ax.axhline(0, color="black", lw=0.7); ax.axvline(0, color="black", lw=0.7)
    ax.set_title(f"{name}\nIC={ic:+.4f}  Signal Sharpe={sh:+.2f}", fontweight="bold")
    ax.set_xlabel("Predicted return"); ax.set_ylabel("Actual return")
    ax.grid(True, alpha=0.2)

# ── Row 1: cumulative signal curves ──────────────────────────────────────────
ax1 = fig.add_subplot(outer[1, :])
y_arr = y_te.values
for name, (preds, col, ic, sh) in models.items():
    strategy = np.sign(preds) * y_arr
    cum      = np.cumsum(strategy)
    ax1.plot(t_test[:len(cum)], cum, color=col, lw=2, label=f"{name} (IC={ic:+.3f})")
bh_cum = np.cumsum(y_arr)   # buy-and-hold NEM
ax1.plot(t_test[:len(bh_cum)], bh_cum, color="black", lw=1.5, ls="--",
         label="Buy-and-hold NEM")
ax1.axhline(0, color="black", lw=0.7)
ax1.set_title("Cumulative signal performance (TEST period, no costs or slippage)",
              fontweight="bold")
ax1.set_ylabel("Cumulative log-return"); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_xlabel("Date")

# ── Row 2: feature importance (Ridge) + IC comparison bar ────────────────────
ax2 = fig.add_subplot(outer[2, 0])
importances = pd.Series(ridge_final.coef_, index=FEATURE_COLS).sort_values(key=abs, ascending=True)
colors_imp  = [C["blue"] if v>0 else C["red"] for v in importances.values]
ax2.barh(importances.index, importances.values, color=colors_imp, alpha=0.8)
ax2.axvline(0, color="black", lw=0.7)
ax2.set_title("Ridge regression feature importance\n(standardised coefficients)",
              fontweight="bold")
ax2.set_xlabel("Coefficient (standardised)")
ax2.grid(True, alpha=0.2, axis="x")

ax3 = fig.add_subplot(outer[2, 1])
names_m = list(models.keys())
ics     = [v[2] for v in models.values()]
shps    = [v[3] for v in models.values()]
x_pos   = np.arange(len(names_m))
bars    = ax3.bar(x_pos, ics, color=[v[1] for v in models.values()], alpha=0.8)
ax3.axhline(0, color="black", lw=0.8)
ax3.axhline(0.05, ls=":", color=C["green"], lw=1.5, label="IC=0.05 (weak threshold)")
ax3.set_xticks(x_pos); ax3.set_xticklabels(names_m, rotation=15, ha="right")
ax3.set_title("Information Coefficient comparison\n(TEST period)", fontweight="bold")
ax3.set_ylabel("IC (Pearson correlation)"); ax3.legend(fontsize=9); ax3.grid(True, alpha=0.2, axis="y")
for bar, sh in zip(bars, shps):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f"Sh={sh:+.2f}", ha="center", fontsize=8)

plt.suptitle("Step F: Walk-Forward Evaluation — TEST Period Only (2019-2024)",
             fontsize=13, fontweight="bold")
plt.savefig("prediction_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Step G — Synthesis and Honest Assessment

### What this pipeline does and does not do

**What it does:**
- Systematically discovers lead-lag relationships in N data streams with proper
  multiple-testing correction (BH FDR)
- Uses lsmash where it is genuinely powerful: computing statistical structure
  distances for regime classification and analog forecasting
- Respects the train/validation/test split throughout — no look-ahead, no data leakage
- Provides interpretable feature importance via the regularised linear model

**What it does NOT do:**
- Guarantee profitable predictions (past lead-lag patterns can and do attenuate)
- Prove causation (statistical lead-lag is not the same as causal influence)
- Work on all pairs at all times (the signals are regime-specific)

### On using lsmash for prediction

The kNN analog approach is the most principled use of SL divergence in a
prediction context. Its information coefficient on the test period tells you
directly whether the statistical structure of past Gold windows is informative
about future NEM returns. Compare this to the Ridge model's IC to understand
how much the lsmash component contributes over purely linear features.

### What would need to change to make this a serious system

1. **Much more data**: weekly equity data back to 1960 via CRSP would give 3,000+
   training weeks, reducing estimation variance substantially.
2. **Out-of-sample p-values**: the current test period is one holdout. Proper
   confidence requires multiple non-overlapping holdouts or bootstrap.
3. **Transaction costs**: any trading strategy evaluation must include realistic
   costs (bid-ask spread, market impact, short-selling costs).
4. **Ensemble**: combine all three models and additional features (options flow,
   sentiment, macroeconomic indicators) for robustness.
5. **Adaptive retraining**: the regime library should be updated as new data arrives.


In [ ]:
print("=" * 65)
print("FINAL PIPELINE SUMMARY")
print("=" * 65)
print()
print(f"Target: {TARGET_COL} next-week return")
print(f"Lead signal: {LEAD_COL} at lag {OPTIMAL_LAG} week(s)")
print(f"Window size: {WINDOW_SIZE} weeks")
print(f"Regime library: {len(lib_seqs)} training windows")
print(f"Test period: {dates_te.iloc[0].date()} to {dates_te.iloc[-1].date()}")
print(f"Test observations: {len(y_te)}")
print()
print("Model performance on TEST (held-out) data:")
print(f"  {'Model':<25s}  {'IC':>8s}  {'RMSE':>10s}  {'Sig.Sharpe':>12s}")
print(f"  {'-'*60}")
for name, (preds, col, ic, sh) in models.items():
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    print(f"  {name:<25s}  {ic:>+8.4f}  {rmse:>10.5f}  {sh:>+12.2f}")
print()
print("Verdict:")
best_model = max(models.items(), key=lambda x: x[1][2])
if best_model[1][2] > 0.05:
    print(f"  {best_model[0]} achieves IC > 0.05 -- a weak but non-trivial signal.")
    print("  This does not imply profitability after costs.")
elif best_model[1][2] > 0:
    print(f"  {best_model[0]} achieves IC = {best_model[1][2]:.4f} -- positive but below")
    print("  conventional trading thresholds. The relationship exists but is")
    print("  too weak to trade reliably without much more data and validation.")
else:
    print("  No model achieves positive IC on the test period.")
    print("  The historical Gold->Newmont relationship has not generalised")
    print("  to the 2019-2024 test period -- consistent with the EMH.")
    print("  This is the most likely honest outcome for any marginal signal.")
print()
print("This is the correct scientific result regardless of its sign.")
print("An honest prediction system reports what it finds, not what it wants to find.")